In [1]:
import boto3
import joblib
import numpy as np
import xarray as xr
import os
import rasterio
import geopandas as gpd
from geotessera import GeoTessera
from rasterio.crs import CRS
from rasterio.io import MemoryFile
import time
import gc
os.environ['AWS_PROFILE'] = 'ctrees'
os.environ['GDAL_DISABLE_READDIR_ON_OPEN'] = 'YES'
os.environ['VSI_CURL_ALLOW_CREDENTIALS'] = 'YES'

In [2]:
s3 = boto3.client('s3')
bucket = 'ctrees-development'
prefix = 'activity/tessera/logging_pantropical_v1/'

In [14]:
# download model
model_name = "pantropical_v1_reinforced"
# model_name = "pantropical_v1"
key = f'{prefix}model/{model_name}.pkl'
local_model_path = f'{model_name}.pkl'
print(f"Downloading model from s3://{bucket}/{key} to {local_model_path}")

s3.download_file(bucket, key, local_model_path)

In [15]:
testing_tiles_path = 's3://ctrees-development/activity/tessera/planet_nicfi_logging_tessera_pantropical_v1_testing.gpkg'
testing_tiles = gpd.read_file(testing_tiles_path)
tiles_id = testing_tiles['name'].tolist()
print(f"There are {len(tiles_id)} testing tiles.")

There are 10 testing tiles.


In [16]:
def save_dataArray_from_memory_to_s3(
    data,
    data_type,
    crs,
    transform,
    nodata,
    bucket,
    output_key
):
    """
    Save an xarray DataArray to S3 as a Cloud Optimized GeoTIFF.    
    Args:
        data (xarray.DataArray): The data to save
        output_key (str): The S3 key for saving the file
        transform (Affine): The geotransform
        crs: The coordinate reference system
    """
    bands, height, width = data.shape
    with MemoryFile() as memfile:
        # Metadata
        profile = {
            'driver': 'COG',
            'dtype': data_type,
            'count': bands,
            'width': width,
            'height': height,
            'crs': crs,
            'transform': transform,
            'compress': 'DEFLATE',
            # 'compress': 'lzw',
            'nodata': nodata
        }

        with memfile.open(**profile) as dataset:
            dataset.write(data)
        memfile.seek(0)
        s3 = boto3.client('s3')
        s3.put_object(Bucket= bucket, Key=output_key, Body=memfile.read())
        print(f'Image saved to S3: s3://{bucket}/{output_key}')


In [17]:
gt = GeoTessera()

In [19]:
# years = range(2017, 2024)
years = [2024]
t0 = time.time()
for year in years:
    for tile_id in tiles_id:
        print(f"Processing tile {tile_id}...")
        geom_tile = testing_tiles[testing_tiles['name'] == tile_id].geometry.values[0]
        bbox = geom_tile.bounds
        print(f"The bounding box (minx, miny, maxx, maxy) is: {bbox}")
        tiles_to_fetch = gt.registry.load_blocks_for_region(bounds=bbox, year=year)
        print(f"Number of tiles to fetch for year {year}: {len(tiles_to_fetch)}")

        for tile in tiles_to_fetch:
            
            t0_tile = time.time()
            lon = tile[1]
            lat = tile[2]
            year = tile[0]
            print(f"Fetching embedding for tile at (lon={lon}, lat={lat}, year={year})")
            embedding, crs, transform = gt.fetch_embedding(lon=lon, lat=lat, year=year)
            print(f"Shape: {embedding.shape}")  # e.g., (1200, 1200, 128)
            print(f"CRS: {crs}")  # Coordinate reference system from landmask

            E = embedding.shape[-1]
            X = embedding.shape[0] 
            Y = embedding.shape[1]
            embeddings_flat = embedding.reshape(-1, E)  # shape (X*Y, 128), each row = one pixel's 128 embeddings

            model = joblib.load(local_model_path)
            probs = model.predict(embeddings_flat)# probability of class 1 only because is binary classifcation
            # cls = (probs > 0.5).astype(np.uint8)
            # cls_reshaped = cls.reshape(X, Y)
            probs_reshaped = probs.reshape(X, Y)

            # fix dimension for rasterio (bands, height, width)
            if probs_reshaped.ndim == 2:
                probs_reshaped = probs_reshaped[np.newaxis, :, :]
            print(f"Shape after adding band dimension: {probs_reshaped.shape}")  # Should be (1, X, Y)

            # Ensure CRS is rasterio-compatible
            crs_obj = CRS.from_user_input(crs)
            nodata = 255
            output_tif = f'tile_{lon}_{lat}_{year}_p.tif'

            # export to s3
            bucket = 'ctrees-development'
            output_key = f'activity/tessera/logging_pantropical_v1/predictions/pantropical_v1_reinforced/tile_{lon}_{lat}/{output_tif}'
            save_dataArray_from_memory_to_s3(
                data=probs_reshaped,
                data_type=probs_reshaped.dtype,
                crs=crs_obj,
                transform=transform,
                nodata=nodata,
                bucket=bucket,
                output_key=output_key
            )

            # remove local file if exists
            working_dir = os.getcwd()
            embedding_folder = os.path.join(working_dir, "global_0.1_degree_representation")
            landmark_folder = os.path.join(working_dir, "global_0.1_degree_tiff_all")
            if os.path.exists(embedding_folder):
                import shutil
                shutil.rmtree(embedding_folder)
                print(f"Removed embedding folder: {embedding_folder}")
            else:
                print(f"Embedding folder not found, skipping removal: {embedding_folder}")


            if os.path.exists(landmark_folder):
                import shutil
                shutil.rmtree(landmark_folder)
                print(f"Removed landmark folder: {landmark_folder}")
            else:
                print(f"Landmark folder not found, skipping removal: {landmark_folder}")
            
            # garbage collection
            del embedding
            del probs
            gc.collect()
            t1_tile = time.time()
            print(f"Processing time for tile {tile_id}: {t1_tile - t0_tile} seconds")

t1 = time.time()
print(f"Total processing time: {t1 - t0} seconds")


Processing tile 0705-1003...
The bounding box (minx, miny, maxx, maxy) is: (-56.07421874215568, -3.68885514257933, -55.8984374921801, -3.513421045096576)
Number of tiles to fetch for year 2024: 6
Fetching embedding for tile at (lon=-56.05, lat=-3.65, year=2024)
Shape: (1107, 1112, 128)
CRS: EPSG:32721
Shape after adding band dimension: (1, 1107, 1112)
Image saved to S3: s3://ctrees-development/activity/tessera/logging_pantropical_v1/predictions/pantropical_v1_reinforced/tile_-56.05_-3.65/tile_-56.05_-3.65_2024_p.tif
Removed embedding folder: /home/ubuntu/tessera_test/notebooks/global_0.1_degree_representation
Removed landmark folder: /home/ubuntu/tessera_test/notebooks/global_0.1_degree_tiff_all
Processing time for tile 0705-1003: 18.112204551696777 seconds
Fetching embedding for tile at (lon=-55.95, lat=-3.65, year=2024)
Shape: (1107, 1112, 128)
CRS: EPSG:32721
Shape after adding band dimension: (1, 1107, 1112)
Image saved to S3: s3://ctrees-development/activity/tessera/logging_pantro